# House Prices: Minimalist Gradient Boosting with YDF

This notebook provides a high-scoring baseline for the House Prices competition using **Yggdrasil Decision Forests (YDF)**. 

The goal is to achieve a good score with minimal code by leveraging the native capabilities of modern tree-based models, avoiding complex manual preprocessing pipelines.

### Key Strategies used in this Notebook:

1.  **Native Handling of Data**: 
    * We use **YDF** (Yggdrasil Decision Forests), which handles missing values (NaNs) and categorical strings natively. 
    * *Why?* This avoids the risks of "imputation noise" (filling missing values with bad guesses) and eliminates the need for One-Hot Encoding.


2.  **Target Transformation (Log-Scale)**: 
    * We train on `log1p(SalePrice)` rather than raw prices.
    * *Why?* The competition evaluates on **RMSLE** (Logarithmic Error). Compressing the target range helps the model care about "relative" errors (percentages) rather than absolute dollar amounts.


3.  **Feature Engineering**:
    * Created a new feature `TotalSF` = `TotalBsmtSF` + `1stFlrSF` + `2ndFlrSF`.
    * *Why?* Total living area is the strongest predictor of price, but it is split across three columns in the raw data. Summing them helps the trees find the signal faster.


4.  **Gradient Boosted Trees (GBT) with Regularization**:
    * We switched from Random Forest to **Gradient Boosted Trees** for higher precision.
    * **Hyperparameters**: We use a low learning rate (`shrinkage=0.05`) combined with `subsample=0.65`.
    * *Why?* This slows down the learning process, preventing the model from memorizing the training data (overfitting) and improving its performance on unseen test data.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import ydf  # Import the new library

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [2]:
# Load the dataset
train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [3]:
# Save IDs for submission
test_ids = test_df['Id']

In [4]:
# Drop 'Id' columns
train_df = train_df.drop("Id", axis=1)
test_df = test_df.drop("Id", axis=1)

In [5]:
# Feature engineering: Create "TotalSF" 

train_df['TotalSF'] = train_df['TotalBsmtSF'] + train_df['1stFlrSF'] + train_df['2ndFlrSF']
test_df['TotalSF'] = test_df['TotalBsmtSF'] + test_df['1stFlrSF'] + test_df['2ndFlrSF']

In [6]:
#Log-Transform the Target
#Use log1p (log(1+x)) to handle the prices
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

In [9]:
# Automated Hyperparameter Tuning ---

# shrinkage=0.05: Learns slower but more precisely (default is usually 0.1)
# num_trees=500: Gives it more time to learn (since we slowed it down)
learner = ydf.GradientBoostedTreesLearner(
    label="SalePrice", 
    task=ydf.Task.REGRESSION,
    num_trees=500,
    shrinkage=0.05,
    subsample=0.65  # Use only 65% of data for each tree to add variety
)

# Train the model

print("Training started... this will take a few minutes...")
model = learner.train(train_df)

Training started... this will take a few minutes...
Feature Utilities is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature PoolQC is a CATEGORICAL feature with an empty dictionary. The feature will not be useful during model training.
Feature MiscFeature is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Train model on 1460 examples
Model trained in 0:00:04.242787


In [11]:
#Predict & Submit
preds = model.predict(test_df)
preds = np.expm1(preds) # Convert back to real dollars

output = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
output.to_csv('submission_v3.csv', index=False)
print("Success! Version 3 submission created.")

Success! Version 3 submission created.
